## NMF-PY Workflow

The steps in this notebook are intended to replicate the preprocessing, base model building, and base model post-processing steps of PMF5. 

The error estimation functionality has not yet been implemented in the new code base.

In [ ]:
# Notebook imports
import os
import sys
import json

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

#### Sample Dataset
The three sample datasets from PMF5 are available for use, but a new dataset can be used in their place.

In [ ]:
# Baton Rouge Dataset
br_input_file = os.path.join("data", "Dataset-BatonRouge-con.csv")
br_uncertainty_file = os.path.join("data", "Dataset-BatonRouge-unc.csv")
br_output_path = os.path.join("data", "output", "BatonRouge")
# Baltimore Dataset
b_input_file = os.path.join("data", "Dataset-Baltimore_con.txt")
b_uncertainty_file = os.path.join("data", "Dataset-Baltimore_unc.txt")
b_output_path = os.path.join("data", "output", "Baltimore")
# Saint Louis Dataset
sl_input_file = os.path.join("data", "Dataset-StLouis-con.csv")
sl_uncertainty_file = os.path.join("data", "Dataset-StLouis-unc.csv")
sl_output_path = os.path.join("data", "output", "StLouis")

#### Code Imports

In [ ]:
from esat.data.datahandler import DataHandler
from esat.model.nmf import NMF
from esat.model.batch_nmf import BatchNMF
from esat.data.analysis import ModelAnalysis

#### Input Parameters

In [ ]:
index_col = "Date"                  # the index of the input/uncertainty datasets
factors = 6                         # the number of factors
method = "ls-nmf"                   # "ls-nmf", "ws-nmf"
models = 20                         # the number of models to train
init_method = "col_means"           # default is column means "col_means", "kmeans", "cmeans"
init_norm = True                    # if init_method=kmeans or cmeans, normalize the data prior to clustering.
seed = 42                           # random seed for initialization
max_iterations = 20000              # the maximum number of iterations for fitting a model
converge_delta = 0.1                # convergence criteria for the change in loss, Q
converge_n = 10                    # convergence criteria for the number of steps where the loss changes by less than converge_delta
verbose = True                      # adds more verbosity to the algorithm workflow on execution.
optimized = False                    # use the Rust code if possible
parallel = True                     # execute the model training in parallel, multiple models at the same time

#### Dataset Selection
One of the three sample datasets can be selected or a new cleaned dataset can be used. Datasets should be cleaned, containing no missing data (either dropping missing/NaNs, or interpolating the missing values).

In [ ]:
# Loading the Baton Rouge dataset
dataset = "br"

In [ ]:
if dataset == "br":
    input_file = br_input_file
    uncertainty_file = br_uncertainty_file
    output_path = br_output_path
elif dataset == "b": 
    input_file = b_input_file
    uncertainty_file = b_uncertainty_file
    output_path = b_output_path
else:
    input_file = sl_input_file
    uncertainty_file = sl_uncertainty_file
    output_path = sl_output_path

#### Load Data
Assign the processed data and uncertainty datasets to the variables V and U. These steps will be simplified/streamlined in a future version of the code.

In [ ]:
data_handler = DataHandler(
    input_path=input_file,
    uncertainty_path=uncertainty_file,
    index_col=index_col
)
V = data_handler.input_data_processed               # Cleaned input dataset (numpy array)
U = data_handler.uncertainty_data_processed         # Cleaned uncertainty dataset (numpy array)

#### Input/Uncertainty Data Metrics and Visualizations

In [ ]:
# Show the input data metrics, including signal to noise ratio of the data and uncertainty
data_handler.metrics

In [ ]:
# Concentration / Uncertainty Scatter plot for specific feature
data_handler.data_uncertainty_plot(feature_idx=2)

In [ ]:
# Species Concentration plot comparing features
data_handler.feature_data_plot(x_idx=0, y_idx=1)

In [ ]:
# Species Timeseries
data_handler.feature_timeseries_plot(feature_selection=[0, 1, 2, 3])

#### Train Model

In [ ]:
%%time
# Training multiple models, optional parameters are commented out.
nmf_models = BatchNMF(V=V, U=U, factors=factors, models=models, method=method, seed=seed, max_iter=max_iterations,
                    # init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n, 
                    parallel=parallel, optimized=optimized,
                    # verbose=verbose
                   )
nmf_models.train()

In [ ]:
%%time
# Training multiple models, optional parameters are commented out.
nmf_models2 = BatchNMF(V=V, U=U, factors=factors, models=models, method=method, seed=seed, max_iter=max_iterations,
                    # init_method=init_method, init_norm=init_norm,
                    converge_delta=converge_delta, converge_n=converge_n,
                    robust_mode=True, robust_n=500, robust_alpha=4,
                    parallel=parallel, optimized=optimized,
                    # verbose=verbose
                   )
nmf_models2.train()

In [ ]:
# Selet the best performing model to review
best_model = nmf_models.best_model
nmf_model = nmf_models.results[best_model]
best_model

In [ ]:
# Initialize the Model Analysis module
model_analysis = ModelAnalysis(datahandler=data_handler, model=nmf_model, selected_model=best_model)

In [ ]:
abs_threshold = 3.0
threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)

In [ ]:
print(f"List of Absolute Scaled Residual Greather than: {abs_threshold}. Count: {threshold_residuals.shape[0]}")
threshold_residuals

In [ ]:
model_analysis.calculate_statistics()
nmf_SE = model_analysis.statistics[["SE"]]
model_analysis.statistics

In [ ]:
model_analysis.plot_estimated_observed(feature_idx=1)

In [ ]:
import numpy as np

factor_i = None
feature_idx = 1
percentage = False

factor_matrices = []
percent_matrices = []
for f in range(nmf_model.factors):
    fW = nmf_model.W[:, f]
    fW = fW.reshape(len(fW), 1)
    fH = nmf_model.H[f]
    f_matrix = np.multiply(fW, fH)
    f_matrix[f_matrix < 1e-8] = 1e-5
    factor_matrices.append(f_matrix)
    percent_matrices.append(f_matrix / nmf_model.V)

z_title = "Percentage (%)" if percentage else "Mass"

_y = data_handler.input_data.index

In [ ]:
x_labels = []
x_label_values = []

if factor_i is None:
    trace_name = data_handler.features[feature_idx]
    plot_title = f"{trace_name} Concentrations for All Factors"
    _z = []
    for i in range(len(factor_matrices)):
        i_z = percent_matrices[i][:, feature_idx] if percentage else factor_matrices[i][:, feature_idx]
        _z.append(i_z)
    _x = [f"Factor {i}" for i in range(1, nmf_model.factors+1)]
    _z = np.array(_z).T
    x_labels = _x
    x_label_values = _x
else:
    plot_title = f"Feature Concentration for Factor {factor_i+1}"
    trace_name = f"Factor {factor_i+1}"
    _z = percent_matrices[factor_i] if percentage else factor_matrices[factor_i]
    _z[_z < 1e-4] = np.nan
    _x = data_handler.features
    
    for f in range(nmf_model.n):
        if not all(np.isnan(_z[:,f])):
            x_labels.append(_x[f])
            x_label_values.append(f)
len(x_labels)

In [ ]:
import plotly.graph_objects as go

matrix_plot = go.Figure()
matrix_plot.add_trace(go.Surface(x=_x, y=_y, z=_z, opacity=1.0, name=trace_name, showscale=True, showlegend=False, colorscale='spectral'))
matrix_plot.update_layout(scene = dict(
    xaxis=dict(title="", nticks=len(x_labels), ticktext=x_labels, tickvals=x_label_values, ),
    yaxis=dict(title=""),
    zaxis=dict(title=f'Concentration {z_title}')
), title=plot_title, width=1200, height=1200,
                          margin=dict(l=65, r=50, b=65, t=60))
matrix_plot.show()

In [ ]:
categories = data_handler.features
profile_p = nmf_model.H / nmf_model.H.sum(axis=0)

profile_radar = go.Figure()
for f in range(nmf_model.factors):
    fH = profile_p[f]
    profile_radar.add_trace(go.Scatterpolar(
        r=fH,
        theta=categories,
        fill='toself',
        name=f"Factor {f+1}",
        hoverinfo="all",
        mode="lines+markers+text"
    ))
profile_radar.update_layout(title="Factor Profile Composition", showlegend=True, width=1400, height=1200,
                            polar=dict(radialaxis=dict(visible=True,range=[0,1])),
    )
profile_radar.show()

In [ ]:
# Imports for comparing to PMF5 outputs
from tests.factor_comparison import FactorComp
from esat.utils import calculate_Q

In [ ]:
if dataset == "br":
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"br{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"br{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"br{factors}f_residuals.txt")
elif dataset == "b":
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"b{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"b{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"b{factors}f_residuals.txt")
else:
    pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"sl{factors}f_profiles.txt")
    pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"sl{factors}f_contributions.txt")
    pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test",
                                              f"sl{factors}f_residuals.txt")
factor_comp = FactorComp(nmf_output_file=None, pmf_profile_file=pmf_profile_file,
                                    pmf_contribution_file=pmf_contribution_file, factors=factors,
                                    features=data_handler.features, residuals_path=pmf_residuals_file)

In [ ]:
pmf_est_V = None
for factor, wh in factor_comp.pmf_WH.items():
    if pmf_est_V is None:
        pmf_est_V = wh
    else:
        pmf_est_V += wh

In [ ]:
model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold)
threshold_residuals = model_analysis.plot_residual_histogram(feature_idx=0, abs_threshold=abs_threshold, est_V=pmf_est_V)

In [ ]:
model_analysis.calculate_statistics(results=pmf_est_V)
model_analysis.statistics

In [ ]:
model_analysis.plot_factor_profile(factor_idx=1)

In [ ]:
factors_data = model_analysis.model.H
normalized_factors_data = factors_data / factors_data.sum(axis=0)
normalized_factors_data[:].shape

In [ ]:
import pandas as pd
factors_contr = model_analysis.model.W
normalized_factors_contr = factors_contr / factors_contr.sum(axis=0)
contr_df = pd.DataFrame(normalized_factors_contr, columns=[f"Factor {i}" for i in range(normalized_factors_contr.shape[1])])
contr_df

In [ ]:
import numpy as np
factors_data = model_analysis.model.H
normalized_factors_data = factors_data / factors_data.sum(axis=0)
normalized_factors_data.shape

In [ ]:
factor_labels = [f"Factor {i}" for i in range(1, factors+1)]
pmf_H = factor_comp.pmf_profiles_df[factor_labels].values.T
pmf_W = factor_comp.pmf_contribution_df[factor_labels].values
# pmf_WH = np.matmul(pmf_W, pmf_H)

In [ ]:
# model_analysis.plot_factor_profile(factor_idx=0, H=pmf_H, W=pmf_W, WH=pmf_WH)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

factor_i = 2
W = pmf_W[:, factor_i]
H = pmf_H[factor_i]
H_sum = pmf_H.sum(axis=0)
factor_matrix = np.matmul(W.reshape(len(W), 1), [H])

factor_conc_sum = factor_matrix.sum(axis=0)
factor_conc_sum[factor_conc_sum == 0] = 1e-12

norm_H = 100 * (H / H_sum)

fig = make_subplots(specs=[[{"secondary_y": True}]], rows=1, cols=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=norm_H, mode="markers", marker=dict(color='red'), name="% of Features"), secondary_y=True, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=factor_conc_sum, marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.6, name='Conc. of Features'), secondary_y=False, row=1, col=1)
fig.update_layout(width=1200, height=600)
fig.update_yaxes(type="log", secondary_y=False, range=[0, np.log10(factor_conc_sum).max()])
fig.update_yaxes(secondary_y=True, range=[0, 100])
fig.show()


In [ ]:
model_analysis.plot_factor_profile(factor_idx=2, H=pmf_H, W=pmf_W)

In [ ]:
import copy
factor_label = f"Factor {factor_i + 1}"
# norm_contr = (W - W.mean()) / W.std()
norm_contr = W / W.mean()
data_df = cosrc.copy(data_handler.input_data)
data_df[factor_label] = norm_contr
data_df.index = pd.to_datetime(data_df.index)
data_df = data_df.sort_index()
data_df = data_df.resample('D').mean()

fig = go.Figure(go.Scatter(x=data_df.index, y=data_df[factor_label], mode='lines+markers'))
fig.update_layout(width=1200, height=800)
fig.show()

In [ ]:
V = data_handler.input_data_processed               # Cleaned input dataset (numpy array)
U = data_handler.uncertainty_data_processed         # Cleaned uncertainty dataset (numpy array)

pmf_H = pmf_H
pmf_W = pmf_W

In [ ]:
pmf_WH = np.matmul(pmf_W, pmf_H)
pmf_residuals = V - pmf_WH

In [ ]:
q2 = np.abs(pmf_residuals/U)
q2_4 = np.abs(pmf_residuals/(4*U))
t = q2.mean() + (q2.std()*3)
q2[q2 > 4] = q2_4[q2 > 4]
Q_robust_pmf = np.sum(np.square(q2), where=q2 < t)
print(f"Q(robust): {Q_robust_pmf}, threshold: {t}")

In [ ]:
Q_true = np.sum(np.square(pmf_residuals / U))
print(f"Q(true): {Q_true}")

In [ ]:
%%time
alpha = 4.0
q3 = np.abs(pmf_residuals/U)
q3_4 = np.abs(pmf_residuals)/((np.sqrt(np.abs(pmf_residuals/U/alpha))*U))
q3[q3 > 4] = q3_4[q3 > 4]
Q_robust_pmf_2 = np.sum(np.square(q3))
print(f"Q(robust): {Q_robust_pmf_2}")

In [ ]:
%%time
alpha = 4.0
scaled_residuals = np.abs(pmf_residuals/U)
robust_U = np.sqrt(scaled_residuals/alpha) * U
robust_residuals = np.abs(pmf_residuals / robust_U)
scaled_residuals[scaled_residuals > 4] = robust_residuals[scaled_residuals > 4]
Q_robust = np.sum(np.square(scaled_residuals))
Q_robust

In [ ]:
np.mean(V/U)

In [ ]:
np.mean(pmf_residuals/U)

### FPeak
FPeak test for rotational ambiguity in the solution. For reference, G: factor contributions (W), and F: factor profile (H)
From the PMF5 users's guide:

$ G^{*} = GT $  and  $ F^{*} = T^{-1}F $  

in NMF  

$ W^{*} = WT $  and  $ H^{*} = T^{-1}H $.

Where $T$ is a nonsingular matrix of size $ p x p $, $p$ is the number of factors. The FPeak runs are specified by a value for the transformation matrix T, both factor matrices are modified and then rerun to convergence. The metrics of each FPeak run that are used to compare to the base model are: dQ(Robust), Q(Robust), % dQ(Robust), Q(Aux) - not applicable here, Q(True) and Converged. The FPeak value cannot be 0 but can be any other positive or negative value.

The impact of the FPeak value is described in the user's guide as:
* positive FPeak values sharpen the factor profile matrix and smear the factor contribution matrix
* negative FPeak values smear the factor profile matix and sharpen the factor contribution matrix

Notes: 
1. A matrix can be tested for nonsingularity by checking if the determinant of that matrix is nonzero.
2. New elements H matrix cannot be negative. The user's guide states 'a pure rotation is only possible none of the lements of the new matrices are less than zero', but the non-negativity constraint in PMF5 (and WS-NMF) is only for the factor profile matrix and uncertainty matrix.
3. The $H^{*}$ matrix must be adjusted to correct negative values. Testing 1 million random configurations of $T$ produced no matices that were both nonsingular and resulted in a nonnegative $H^{*}$.




In [ ]:
from esat.utils import q_loss, qr_loss
from tqdm import trange

We = nmf_model.We
H = nmf_model.H
W = nmf_model.W
Qm0 = nmf_model.Qrobust
S = np.ones(shape=W.shape[0]) * 0.1
max_i = 5000
converge_d = 1e-3
converge_n = 20

In [ ]:
def qaux_loss(W, Wp, D, S):
    r = np.square(W + D - Wp)
    qaux = np.divide(r.sum(axis=1), np.square(S))
    return np.sum(qaux)

def ls_nmf_w(V, We, W, H):
    WeV = np.multiply(We, V)
    WH = np.matmul(W, H)
    W_num = np.matmul(WeV, H.T)
    W_den = np.matmul(np.multiply(We, WH), H.T)
    W = np.multiply(W, np.divide(W_num, W_den))
    return W

def ls_nmf_h(V, We, W, H):
    WeV = np.multiply(We, V)
    WH = np.matmul(W, H)
    H_num = np.matmul(W.T, WeV)
    H_den = np.matmul(W.T, np.multiply(We, WH))
    H = np.multiply(H, np.divide(H_num, H_den))
    return H

def ls_nmf(V, We, W, H):
    WeV = np.multiply(We, V)
    WH = np.matmul(W, H)
    H_num = np.matmul(W.T, WeV)
    H_den = np.matmul(W.T, np.multiply(We, WH))
    H = np.multiply(H, np.divide(H_num, H_den))

    W_num = np.matmul(WeV, H.T)
    W_den = np.matmul(np.multiply(We, WH), H.T)
    W = np.multiply(W, np.divide(W_num, W_den))
    return H, W

In [ ]:
# Fpeak - Model Runs
# f_peaks = [0.1, -0.1, 0.25, -0.25, 0.5, -0.5, 0.75, -0.75, 1.0, -1.0, 1.5, -1.5, 2.0, -2.0]
f_peaks = [0.5, -0.5, 1.0, -1.0]
fp_results = {}
single_step = False
for fp in f_peaks:
    phi = np.full(shape=(factors, factors), fill_value=fp)
    for i in range(factors):
        phi[i,i] = 0.0
    t_iter = trange(max_i, desc=f"W Update - Q(base): {Qm0}, Q(main): NA, Q(aux): NA", position=0, leave=True)
    W_i = W
    H_i = H
    qa_list = []
    qm_list = []
    qd_list = []
    for i in t_iter:
        WtW = np.matmul(W_i.transpose(), W_i)
        D = np.matmul(np.matmul(W_i, np.linalg.inv(WtW)), phi)
        W_d = D + W_i
        W_d[W_d < 0.0] = 0.0
        W_i = ls_nmf_w(V=V, We=We, W=W_d, H=H_i)
        Qm_i = q_loss(V=V, U=U, W=W_i, H=H_i)
        Qmr_i, _ = qr_loss(V=V, U=U, W=W_i, H=H_i)
        Qaux_i = qaux_loss(W=W, Wp=W_i, D=D, S=S)
        Qm = Qm_i + Qaux_i
        t_iter.set_description(f"W Update - Q(base): {round(Qm0,4)}, Q(main): {round(Qm,4)}, Q(aux): {round(Qaux_i,4)}")
        qa_list.append(Qaux_i)
        qm_list.append(Qm)
        if len(qa_list) > converge_n:
            qa_list.pop(0)
            qm_list.pop(0)
            qd_list.append(qm_list[-1] - qm_list[-2])
            if len(qd_list) > converge_n:
                qd_list.pop(0)
            if np.abs(qa_list[0] - qa_list[-1]) <= converge_d:
                break
    t_iter = trange(max_i, desc=f"H Update - Q(True): NA, Q(Robust): NA", position=0, leave=True)
    qh_list = []
    mlist = []
    for i in t_iter:
        H_i = ls_nmf_h(V=V, We=We, W=W_i, H=H_i)
        Qm_i = q_loss(V=V, U=U, W=W_i, H=H_i)
        Qmr_i, _ = qr_loss(V=V, U=U, W=W_i, H=H_i)
        qh_list.append(Qm_i)
        mlist.append(Qm_i)
        t_iter.set_description(f"H Update - Q(True): {round(Qm_i,2)}, Q(Robust): {round(Qmr_i,2)}")
        if len(qh_list) > converge_n:
            qh_list.pop(0)
            if np.abs(qh_list[0] - qh_list[-1]) <= 1e-2:
                break
    Qm_2 = q_loss(V=V, U=U, W=W_i, H=H_i)
    Qr_2, _ = qr_loss(V=V, U=U, W=W_i, H=H_i)
    fp_results[str(fp)] = {
        'W': W_i,
        'H': H_i,
        'Q(True)': Qm_2,
        'Q(Robust)': Qr_2,
        'Strength': str(fp),
        'Q(Aux)': Qaux_i,
        'Q(M)': Qm,
        'Q_list': mlist
    }    
    print(f"Strength: {fp}, dQ(Robust): {round(Qmr_i - Qm0, 2)}, Q(Robust): {round(Qmr_i,2)}, % dQ(Robust): {round(100*((Qmr_i/Qm0)-1),2)}, Q(Aux): {round(Qaux_i,2)}, Q(True): {round(Qm_i,2)}, Q2: {round(Qm_2, 2)}")

In [ ]:
Q_plot = go.Figure()
Q_plot.add_trace(go.Scatter(x=list(range(len(fp_results['1.0']['Q_list']))), y=fp_results['1.0']['Q_list']))
Q_plot.update_layout(height=800, width=800)
Q_plot.show()

In [ ]:
# Fpeak - Profiles/Contributions

factor_i = 1
fpeak_i = '1.0'

selected_fpeak = fp_results[fpeak_i]
b_W = nmf_model.W[:, factor_i]
b_H = nmf_model.H[factor_i]
b_H_sum = nmf_model.H.sum(axis=0)
b_factor_matrix = np.matmul(b_W.reshape(len(b_W), 1), [b_H])

b_factor_conc_sum = b_factor_matrix.sum(axis=0)
b_factor_conc_sum[b_factor_conc_sum == 0.0] = 1e-12

i_W = selected_fpeak['W'][:, factor_i]
i_H = selected_fpeak['H'][factor_i]
i_H_sum = selected_fpeak['H'].sum(axis=0)
i_factor_matrix = np.matmul(i_W.reshape(len(i_W), 1), [i_H])

i_factor_conc_sum = i_factor_matrix.sum(axis=0)
i_factor_conc_sum[i_factor_conc_sum == 0] = 1e-12

b_norm_H = np.round(100 * (b_H / b_H_sum),2)
i_norm_H = np.round(100 * (i_H / i_H_sum),2)

fig = make_subplots(specs=[[{"secondary_y": True}]], rows=1, cols=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=b_norm_H, mode="markers", marker=dict(color='gray'), name="Base % of Features", opacity=0.8), secondary_y=True, row=1, col=1)
fig.add_trace(go.Scatter(x=data_handler.features, y=i_norm_H, mode="markers", marker=dict(color='red'), name="FPeak % of Features", opacity=0.6), secondary_y=True, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=b_factor_conc_sum, marker_color='rgb(203,203,203)', marker_line_color='rgb(186,186,186)', marker_line_width=1.5, opacity=0.6, name='Base Conc. of Features'), secondary_y=False, row=1, col=1)
fig.add_trace(go.Bar(x=data_handler.features, y=i_factor_conc_sum, marker_color='rgb(134,236,168)', marker_line_color='rgb(125,220,157)', marker_line_width=1.5, opacity=0.6, name='FPeak Conc. of Features'), secondary_y=False, row=1, col=1)
fig.update_layout(width=1200, height=600, title=f"Fpeak Factor Profile - FP={fpeak_i} - Factor {factor_i}", barmode='group', scattermode='group', hovermode="x unified")
fig.update_yaxes(type="log", secondary_y=False, range=[0, np.log10(b_factor_conc_sum).max()], row=1, col=1)
fig.update_yaxes(secondary_y=True, range=[0, 100])
fig.show()

b_norm_contr = b_W / b_W.mean()
b_data_df = cosrc.copy(data_handler.input_data)
b_data_df[factor_label] = b_norm_contr
b_data_df.index = pd.to_datetime(b_data_df.index)
b_data_df = b_data_df.sort_index()
b_data_df = b_data_df.resample('D').mean()

i_norm_contr = i_W / i_W.mean()
i_data_df = cosrc.copy(data_handler.input_data)
i_data_df[factor_label] = i_norm_contr
i_data_df.index = pd.to_datetime(i_data_df.index)
i_data_df = i_data_df.sort_index()
i_data_df = i_data_df.resample('D').mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=b_data_df.index, y=b_data_df[factor_label], mode='lines+markers', marker_color='rgb(186,186,186)', name="Base Factor Contributions"))
fig.add_trace(go.Scatter(x=i_data_df.index, y=i_data_df[factor_label], mode='lines+markers', marker_color='rgb(125,220,157)', name="FPeak Factor Contributions"))
fig.update_layout(width=1200, height=800, title=f"Fpeak Factor Contributions - Fpeak={fpeak_i} - Factor {factor_i}", hovermode="x unified")
fig.update_yaxes(title_text="Factor Contributions")
fig.show()

In [ ]:
# Fpeak - Factor Fingerprints
import plotly.express as px
fpeak_i = '1.0'

selected_fpeak = fp_results[fpeak_i]
b_H = nmf_model.H
fp_H = selected_fpeak['H']

b_normalized = 100 * (b_H / b_H.sum(axis=0))
fp_normalized = 100 * (fp_H / fp_H.sum(axis=0))

fp_factors_fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Base Profile", "Fpeak Profile"), vertical_spacing=0.075)
colors = px.colors.sequential.Viridis_r
for idx in range(factors-1, -1, -1):
    fp_factors_fig.add_trace(go.Bar(name=f"Base Factor {idx+1}", x=data_handler.features, y=b_normalized[idx], marker_color=colors[idx]), row=1, col=1)
    fp_factors_fig.add_trace(go.Bar(name=f"Fpeak Factor {idx+1}", x=data_handler.features, y=fp_normalized[idx], marker_color=colors[idx]), row=2, col=1)
fp_factors_fig.update_layout(title=f"Fpeak Factor Fingerprints - Fpeak={fpeak_i}", width=1200, height=800, barmode='stack', hovermode='x unified')
fp_factors_fig.update_yaxes(title_text="% Feature Concentration", range=[0, 100])
fp_factors_fig.show()

In [ ]:
import plotly.figure_factory as ff
# Fpeak - G-Space Plots
fpeak_i = '1.0'

selected_fpeak = fp_results[fpeak_i]
b_W = nmf_model.W
fp_W = selected_fpeak['W']

b_normalized_factors_contr = b_W / b_W.sum(axis=0)
fp_normalized_factors_contr = fp_W / fp_W.sum(axis=0)

f1_idx = 0
f2_idx = 1
show_base = True
show_delta = True

if show_delta:
    arrows = ((fp_normalized_factors_contr[:, f1_idx] - b_normalized_factors_contr[:, f1_idx]), (fp_normalized_factors_contr[:, f2_idx] - b_normalized_factors_contr[:, f2_idx]))
    fp_g_fig = ff.create_quiver(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], u=arrows[0], v=arrows[1], name="Fpeak Delta", line_width=1, arrow_scale=0.01, scale=0.99)
    fp_g_fig.add_trace(go.Scatter(x=fp_normalized_factors_contr[:, f1_idx],y=fp_normalized_factors_contr[:, f2_idx], mode='markers', name="Fpeak"))
    fp_g_fig.add_trace(go.Scatter(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], mode='markers', name="Base"))
else:    
    fp_g_fig = go.Figure()
    fp_g_fig.add_trace(go.Scatter(x=fp_normalized_factors_contr[:, f1_idx],y=fp_normalized_factors_contr[:, f2_idx], mode='markers', name="Fpeak"))
    if show_base:
        fp_g_fig.add_trace(go.Scatter(x=b_normalized_factors_contr[:, f1_idx],y=b_normalized_factors_contr[:, f2_idx], mode='markers', name="Base"))
fp_g_fig.update_layout(title=f"Fpeak G-Space Plot - Fpeak={fpeak_i}", width=800, height=800)
fp_g_fig.update_yaxes(title_text=f"Factor {f1_idx+1} Contributions (avg=1)")
fp_g_fig.update_xaxes(title_text=f"Factor {f2_idx+1} Contributions (avg=1)")
fp_g_fig.show()

In [ ]:
#Fpeak - Factor Contributions
fpeak_i = '1.0'
contribution_threshold = 0.06
converged = True
feature_idx = 1

x_label = data_handler.input_data.columns[feature_idx]
factors_data = fp_results[fpeak_i]['H']
normalized_factors_data = 100 * (factors_data / factors_data.sum(axis=0))

feature_contr = normalized_factors_data[:, feature_idx]
feature_contr_inc = []
feature_contr_labels = []
feature_legend = {}
for idx in range(feature_contr.shape[0]-1, -1, -1):
    idx_l = idx+1
    if feature_contr[idx] > contribution_threshold:
        feature_contr_inc.append(feature_contr[idx])
        feature_contr_labels.append(f"Factor {idx_l}")
        feature_legend[f"Factor {idx_l}"] = f"Factor {idx_l} = {factors_data[idx:, feature_idx]}"
feature_fig = go.Figure(data=[go.Pie(labels=feature_contr_labels, values=feature_contr_inc, hoverinfo="label+value", textinfo="percent")])
feature_fig.update_layout(title=f"Factor Contributions to Feature: {x_label} - Fpeak={fpeak_i}", width=1200, height=600,
                                  legend_title_text=f"Factor Contribution > {contribution_threshold}%")
feature_fig.show()

factors_contr = fp_results[fpeak_i]['W']
normalized_factors_contr = 100 * (factors_contr / factors_contr.sum(axis=0))
factor_labels = [f"Factor {i}" for i in range(1, normalized_factors_contr.shape[1]+1)]
contr_df = pd.DataFrame(normalized_factors_contr, columns=factor_labels)
contr_df.index = pd.to_datetime(data_handler.input_data.index)
contr_df = contr_df.sort_index()
contr_df = contr_df.resample('D').mean()

contr_fig = go.Figure()
for factor in factor_labels:
    contr_fig.add_trace(go.Scatter(x=contr_df.index, y=contr_df[factor], mode='lines+markers', name=factor))
contr_fig.update_layout(title=f"Factor Contributions (avg=1) - Fpeak={fpeak_i}",
                                width=1200, height=600,
                                legend=dict(orientation="h", xanchor="right", yanchor="bottom", x=1, y=1.02))
contr_fig.update_yaxes(title_text="Normalized Contribution")
contr_fig.show()

In [ ]:
factor_label